# Direct binding

This tutorial determines the dissociation constants of three fluorescent
tracers by fluorescence polarization (FP). Receptor was titrated in 12 two-fold
steps, from 2 nM to 4 µM, into a fixed concentration of tracer, and
polarization was measured in mP. Each tracer was measured in three independent
experiments with technical triplicates. The data are in
{download}`direct-binding.csv <data/direct-binding.csv>`.

The x axis is the total receptor concentration in nM.

In [ ]:
import matplotlib.pyplot as plt

import bindcurve as bc

data = bc.DoseResponseData.from_csv("data/direct-binding.csv", format="wide")
data.summary()

## Tracer far below its Kd

`tracer_a` was used at 1 nM, well below its $K_d$. Tracer binding then hardly
changes the free receptor concentration, so `dir_simple`, which treats the
receptor concentration as free, applies. It needs no assay constants.

In [ ]:
simple = bc.fit(data.select("tracer_a"), "dir_simple")
simple.summary()[["compound_id", "N_fit", "Kds", "Kds_CI95_lower", "Kds_CI95_upper"]]

## Tracer comparable to its Kd

`tracer_b` was used at 10 nM, comparable to its $K_d$. Here tracer binding
lowers the free receptor concentration noticeably, so `dir_specific` is used.
It works with total receptor and needs the total tracer concentration `LsT`.

In [ ]:
specific = bc.fit(data.select("tracer_b"), "dir_specific", fixed={"LsT": 10.0})
specific.summary()[["compound_id", "N_fit", "Kds", "Kds_CI95_lower", "Kds_CI95_upper"]]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
bc.plot_fits(specific, band=True, ax=ax)
ax.set_xlabel("total receptor (nM)")
ax.set_ylabel("polarization (mP)")
ax.legend()
plt.show()

## Tracer with nonspecific binding

`tracer_c`, also at 10 nM, binds nonspecifically as well. A separate
measurement gave a nonspecific-binding factor of `Ns = 0.25`, the ratio of
nonspecifically bound to free tracer. `dir_total` takes it as a fixed value
together with `LsT`.

In [ ]:
total = bc.fit(data.select("tracer_c"), "dir_total", fixed={"LsT": 10.0, "Ns": 0.25})
total.summary()[["compound_id", "N_fit", "Kds", "Kds_CI95_lower", "Kds_CI95_upper"]]

## Results

None of the fits has a quality warning:

In [ ]:
for results in (simple, specific, total):
    print(results.experiments()[["compound_id", "experiment_id", "warnings"]].to_string(index=False))

The summary curves of the three tracers, each from three independent
experiments:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for results, color in zip((simple, specific, total), ("C0", "C1", "C2")):
    bc.plot_compounds(results, colors=color, ax=ax)
ax.set_xlabel("total receptor (nM)")
ax.set_ylabel("polarization (mP)")
ax.legend()
plt.show()

In [ ]:
for results in (simple, specific, total):
    print(results.report(uncertainty="ci95", unit="nM").to_string(index=False))